In [1]:
import pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

from src.preprocessing import preprocess
from src.my_features_260429 import (
    add_features_BEFORE_team_preprocess,
    add_features_AFTER_team_preprocess
)

In [2]:
# 데이터 로드
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

y = train['임신 성공 여부']

In [3]:
train, test = add_features_BEFORE_team_preprocess(train, test)
X_full, X_test = preprocess(train, test)
X_full, X_test = add_features_AFTER_team_preprocess(X_full, X_test)

In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import numpy as np
from scipy.stats import rankdata  # 🔥 추가

seeds = [42, 52, 62]

oof_total = np.zeros(len(X_full))
oof_list = []  # 🔥 추가: seed별 저장

for seed in seeds:
    print(f"\n====== SEED {seed} ======")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof = np.zeros(len(X_full))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full, y)):
        print(f"\n🔥 Fold {fold+1}")

        X_tr, X_va = X_full.iloc[tr_idx], X_full.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = CatBoostClassifier(
            iterations=1200,
            learning_rate=0.025,
            depth=9,
            eval_metric='AUC',
            verbose=0,
            random_seed=seed
        )

        model.fit(
            X_tr, y_tr,
            eval_set=(X_va, y_va),
            early_stopping_rounds=100,
            verbose=0
        )

        oof[va_idx] = model.predict_proba(X_va)[:, 1]

    auc = roc_auc_score(y, oof)
    print(f"\n🔥 SEED {seed} AUC: {auc:.5f}")

    oof_total += oof / len(seeds)
    oof_list.append(oof)  # 🔥 저장

# 🔥 기존 평균 AUC
final_auc = roc_auc_score(y, oof_total)
print("\n🚀 FINAL ENSEMBLE AUC:", final_auc)
# 🔥 rank 앙상블 검증 (여기에 붙이기)
from scipy.stats import rankdata

ranks = [rankdata(oof) for oof in oof_list]
rank_ensemble = np.mean(ranks, axis=0)

rank_auc = roc_auc_score(y, rank_ensemble)
print("\n🔥 RANK ENSEMBLE AUC:", rank_auc)


====== SEED 42 ======

🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

🔥 SEED 42 AUC: 0.73978

====== SEED 52 ======

🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

🔥 SEED 52 AUC: 0.73977

====== SEED 62 ======

🔥 Fold 1

🔥 Fold 2

🔥 Fold 3

🔥 Fold 4

🔥 Fold 5

🔥 SEED 62 AUC: 0.73980

🚀 FINAL ENSEMBLE AUC: 0.7401640964946242

🔥 RANK ENSEMBLE AUC: 0.7401694756935008


In [5]:
# 🔥 FINAL TRAIN (전체 데이터)

from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.025,
    depth=9,
    eval_metric='AUC',
    verbose=200,
    random_seed=42
)

model.fit(X_full, y)

0:	total: 34.5ms	remaining: 41.4s
200:	total: 7.08s	remaining: 35.2s
400:	total: 14.2s	remaining: 28.2s
600:	total: 21s	remaining: 20.9s
800:	total: 27.9s	remaining: 13.9s
1000:	total: 34.9s	remaining: 6.93s
1199:	total: 41.9s	remaining: 0us


CatBoostClassifier(depth=9, eval_metric='AUC', iterations=1200, learning_rate=0.025, random_seed=42, verbose=200)

In [11]:
print(pd.read_csv("data/sample_submission.csv").columns)

Index(['ID', 'probability'], dtype='object')


In [12]:
print(X_full.shape)

(256351, 82)


제출용 테스트시작 260427  CAT + XGB + 팀 최고모델 비율 앙상블

In [13]:
cat_model = CatBoostClassifier(
    iterations=1200,
    learning_rate=0.025,
    depth=9,
    eval_metric='AUC',
    verbose=0,
    random_seed=42
)

cat_model.fit(X_full, y)

test_pred_cat = cat_model.predict_proba(X_test)[:, 1]

print("Cat done")

Cat done


In [14]:
submission = pd.read_csv("data/sample_submission.csv")
submission["probability"] = test_pred_cat

submission.to_csv("submission_final.csv", index=False)